In [1]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score



In [2]:
 df = pd.read_csv("Nassau Candy Distributor (1).csv")
    
  

In [3]:
df["Order Date"] = pd.to_datetime(
    df["Order Date"],
    format="%d-%m-%Y"
)

df["Ship Date"] = pd.to_datetime(
    df["Ship Date"],
    format="%d-%m-%Y"
)

In [4]:
df["Lead_Time"] = (
   df["Ship Date"] - df["Order Date"]
).dt.days

In [5]:
df["Month"] = df["Order Date"].dt.month
df["Weekday"] = df["Order Date"].dt.dayofweek

In [6]:
X = df[
[
"Division",
"Region",
"Ship Mode",
"Product Name",
"Sales",
"Units",
"Gross Profit",
"Cost",
"Month",
"Weekday"
]
]

y = df["Lead_Time"]

In [7]:
cat_cols = [
"Division",
"Region",
"Ship Mode",
"Product Name"
]

num_cols = [
"Sales",
"Units",
"Gross Profit",
"Cost",
"Month",
"Weekday"
]

In [8]:

preprocessor = ColumnTransformer(
[
(
"cat",
OneHotEncoder(handle_unknown="ignore"),
cat_cols
)
],
remainder="passthrough"
)

In [9]:
model = Pipeline([
("prep", preprocessor),
("rf", RandomForestRegressor(
    n_estimators=200,
    random_state=42
))
])

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.2,
random_state=42
)

In [11]:
model.fit(X_train, y_train)


,steps,"[('prep', ...), ('rf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [12]:
pred = model.predict(X_test)

In [13]:
print("MAE =", mean_absolute_error(y_test, pred))
print("R2 =", r2_score(y_test, pred))


MAE = 219.38574588003272
R2 = -0.03322108449670336


In [14]:
pickle.dump(model, open("leadtime_model.pkl", "wb"))

print("Model saved successfully")

Model saved successfully


In [15]:
!pip install streamlit streamlit-jupyter

In [16]:
%%writefile app.py

import streamlit as st
import pandas as pd

st.set_page_config(layout="wide")

st.title("🍭 Nassau Candy Dashboard")

df = pd.read_csv("Nassau Candy Distributor (1).csv")

st.subheader("Dataset")

st.dataframe(df.head())

st.subheader("Sales")

st.bar_chart(
    df.groupby("Product Name")["Sales"].sum()
)

Overwriting app.py


In [17]:
from streamlit_jupyter import StreamlitPatcher

StreamlitPatcher().jupyter()

In [18]:
%run app.py

2026-06-16 16:36:08.658 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


# 🍭 Nassau Candy Dashboard

### Dataset

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90


### Sales

2026-06-16 16:36:16.559 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 16:36:17.946 
  command:

    streamlit run app.py [ARGUMENTS]
2026-06-16 16:36:17.947 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 16:36:17.948 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [25]:
# Import required libraries
import streamlit as st
import plotly.express as px
import pandas as pd

# Load your data (replace with your actual data loading method)
# df = pd.read_csv("your_data.csv")  # Uncomment and modify this line
# For demonstration, creating a sample DataFrame
df = pd.DataFrame({
    'Ship Mode': ['Standard', 'Express', 'Priority', 'Standard'],
    'Lead_Time': [5, 2, 1, 4],
    'Product Name': ['Product A', 'Product B', 'Product C', 'Product D'],
    'Sales': [1000, 1500, 800, 1200],
    'Region': ['East', 'West', 'North', 'South']
})

# Define filtered DataFrame (replace with your actual filtering logic)
filtered = df.copy()  # Modify this based on your filtering requirements

st.subheader("Lead Time by Ship Mode")

ship_df = (
    df.groupby("Ship Mode")["Lead_Time"]
    .mean()
    .reset_index()
)

fig2 = px.pie(
    ship_df,
    names="Ship Mode",
    values="Lead_Time",
    title="Lead Time by Ship Mode"
)

st.plotly_chart(
    fig2,
    use_container_width=True
)

# ====================================================
# TOP PRODUCTS
# ====================================================
st.subheader("Top Products by Sales")

top_products = (
    df.groupby("Product Name")["Sales"]
    .sum()
    .reset_index()
    .sort_values(
        by="Sales",
        ascending=False
    )
    .head(10)
)

fig3 = px.bar(
    top_products,
    x="Sales",
    y="Product Name",
    orientation="h",
    title="Top Products"
)

st.plotly_chart(
    fig3,
    use_container_width=True
)

# ====================================================
# SALES BY REGION
# ====================================================
st.subheader("Sales by Region")

sales_region = (
    df.groupby("Region")["Sales"]
    .sum()
    .reset_index()
)

fig4 = px.bar(
    sales_region,
    x="Region",
    y="Sales",
    color="Sales"
)

st.plotly_chart(
    fig4,
    use_container_width=True
)

# ====================================================
# LEAD TIME DISTRIBUTION
# ====================================================
st.subheader("Lead Time Distribution")

fig5 = px.histogram(
    df,
    x="Lead_Time",
    nbins=20,
    title="Lead Time Distribution"
)

st.plotly_chart(
    fig5,
    use_container_width=True
)

# ====================================================
# FILTERED DATA
# ====================================================
st.subheader("Filtered Data")

st.dataframe(filtered)

# ====================================================
# DOWNLOAD
# ====================================================
csv = filtered.to_csv(index=False)

st.download_button(
    label="Download Filtered Data",
    data=csv,
    file_name="filtered_data.csv",
    mime="text/csv"
)

### Lead Time by Ship Mode

2026-06-16 17:10:55.216 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-06-16 17:10:55.241 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.243 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.248 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.252 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.256 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


### Top Products by Sales

2026-06-16 17:10:55.383 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-06-16 17:10:55.387 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.389 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.391 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.393 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.395 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


### Sales by Region

...

2026-06-16 17:10:55.604 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-06-16 17:10:55.608 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.611 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.616 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.619 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.621 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


### Filtered Data

,Ship Mode,Lead_Time,Product Name,Sales,Region
0,Standard,5,Product A,1000,East
1,Express,2,Product B,1500,West
2,Priority,1,Product C,800,North
3,Standard,4,Product D,1200,South


2026-06-16 17:10:55.764 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.766 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.771 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.774 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.783 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.790 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-16 17:10:55.798 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


False